# OpenEnv Wordle with GRPO using TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huggingface/trl/blob/main/examples/grpo_wordle/grpo_wordle.ipynb)

![trl banner](https://huggingface.co/datasets/trl-lib/documentation-images/resolve/main/trl_banner_dark.png)


With [**Transformers Reinforcement Learning (TRL)**](https://github.com/huggingface/trl), you can train a model that learns to **play Wordle**, a word-guessing game, through interaction and reinforcement.

This notebook covers the GRPO step. It starts from [`openenv/Qwen3-1.7B-Wordle-SFT`](https://huggingface.co/openenv/Qwen3-1.7B-Wordle-SFT), a Qwen3-1.7B that already learned the game with a short SFT warmup. To build that model yourself (the data and the SFT step), follow the [OpenEnv Wordle tutorial](https://huggingface.co/docs/openenv/tutorials/wordle-grpo).

- [TRL GitHub Repository](https://github.com/huggingface/trl) -- star us to support the project!
- [Official TRL Examples](https://huggingface.co/docs/trl/example_overview)
- [Community Tutorials](https://huggingface.co/docs/trl/community_tutorials)
- [OpenEnv](https://github.com/openenv/OpenEnv)


An **agentic environment** is a setting where a model can take actions, observe outcomes, and adjust its behavior based on feedback, similar to how humans learn from trial and error.
In this case, the agent interacts with the **Wordle** environment through the [**OpenEnv**](https://github.com/openenv/OpenEnv) framework, which standardizes multi-agent and RL-style text environments.

[Wordle](https://en.wikipedia.org/wiki/Wordle) is a popular word puzzle where the player must guess a secret five-letter word within six tries.
After each guess, feedback indicates whether each letter is:
- **GREEN (G)**: Correct and in the right position
- **YELLOW (Y)**: Present but in the wrong position
- **GRAY (X)**: Not in the word

This feedback loop makes Wordle a perfect environment for **RL with LLMs**, where the goal is to maximize the probability of guessing the correct word efficiently.


We'll fine-tune a model using **GRPO** (Group Relative Policy Optimization) via TRL.
Using `environment_factory`, the trainer automatically handles:
1. Creating environment instances for each rollout.
2. Generating model completions and parsing tool calls.
3. Stepping through the environment with the model's actions.
4. Collecting rewards and managing the interaction loop.

This means you only need to define the environment class and reward function -- the trainer takes care of the rest.

The SFT warmup comes first because the base model almost never wins, which leaves GRPO little to learn from. In our run, training took about 2 hours 15 minutes on one A100 80GB.


## Install dependencies

We'll start by installing **TRL** (with vLLM support), the **OpenEnv** Wordle environment, and **trackio** for logging.

In [ ]:
!pip install -Uq "trl[vllm]" "openenv-textarena @ git+https://github.com/openenv/OpenEnv.git@v0.10.0#subdirectory=envs/textarena_env" trackio

### Log in to Hugging Face

Log in to your **Hugging Face** account to save your fine-tuned model, track your experiment results directly on the Hub or access gated models. You can find your **access token** on your [account settings page](https://huggingface.co/settings/tokens).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

## Define the system prompt

This prompt instructs the model on how to play Wordle. It includes the game rules, feedback format, and importantly, tells the model to use the `guess` tool to submit guesses. The `environment_factory` pattern uses tool calling to interact with the environment, so the model needs to know which tool to call.

In [ ]:
prompt = """You are an expert Wordle solver with deep knowledge of English vocabulary, letter frequency patterns, and optimal guessing strategies.

Follow these rules to play Wordle:

1. The target is a 5-letter English word
2. You have 6 attempts to guess the correct word
3. After each guess, you receive color-coded feedback:
   - GREEN (G): Letter is correct and in the correct position
   - YELLOW (Y): Letter is in the word but in the wrong position
   - GRAY (X): Letter is not in the word at all
4. All guesses must be valid 5-letter English words
5. You cannot reuse a word you've already guessed
6. Use the tool `guess` to make a guess.
"""

## Define the environment

The `WordleEnv` class wraps the OpenEnv TextArena Wordle environment into the interface expected by `environment_factory`.

When you pass `environment_factory=WordleEnv` to the trainer, it will:
1. Create a new `WordleEnv()` instance for each rollout episode.
2. Call `reset()` to start a new game (returns the initial observation or `None`).
3. Automatically generate model completions, parse tool calls, and invoke the corresponding methods (e.g., `guess(...)`).
4. Repeat until the environment signals `done=True` or the max completion length is reached.

The environment exposes its public methods as tools. Any public method (other than `reset`) with a docstring is automatically discovered and exposed as a callable tool. Here, the `guess` method lets the model submit a Wordle guess and receive feedback.

Each training step plays 64 games, one environment session each. [Duplicate the `openenv/wordle` Space](https://huggingface.co/spaces/openenv/wordle?duplicate=true), set the `MAX_CONCURRENT_ENVS=64` variable on your copy and use its URL below.

The class also tracks what the reward needs: whether the game was won, how many guesses it took, the most green letters in a guess and the number of invalid moves.

For more information, refer to the [TRL-OpenEnv documentation](https://huggingface.co/docs/trl/main/en/openenv).

In [ ]:
import re

from textarena_env import TextArenaAction, TextArenaEnv

env_url = "https://<user>-wordle.hf.space"  # your copy of openenv/wordle
FEEDBACK = re.compile(r"Feedback:\n([A-Z ]+)\n([GYX ]+)")


class WordleEnv:
    def __init__(self):
        self.client = TextArenaEnv(base_url=env_url)

    def reset(self, **kwargs) -> None | str:
        result = self.client.reset()
        # The game returns cumulative feedback each turn (new text appended at the end), so
        # we store the previous full response and slice out only the newly appended part.
        self._last_full_feedback = result.observation.messages[0].content
        self.done = False
        self.won = False
        self.num_guesses = 0
        self.best_greens = 0
        self.invalid_moves = 0
        return self._last_full_feedback

    def guess(self, guess: str) -> str:
        """
        Make a guess in the Wordle environment.

        Args:
            guess: The guessed word, formatted as '[abcde]'

        Returns:
            The feedback message from the environment.
        """
        if self.done:
            raise ValueError("Game over.")
        result = self.client.step(TextArenaAction(message=guess))
        _full_feedback = result.observation.messages[0].content
        # Just take the new feedback since the last guess
        feedback = _full_feedback[len(self._last_full_feedback):]
        self._last_full_feedback = _full_feedback
        if "invalid move" in feedback.lower():
            self.invalid_moves += 1
        else:
            self.num_guesses += 1
            match = FEEDBACK.search(feedback)
            if match:
                self.best_greens = max(self.best_greens, match.group(2).count("G"))
            self.won = result.done and result.reward >= 1.0
        self.done = result.done
        return feedback

## Define the reward functions

The environment pays a win the same at any speed and pays partial credit only at the end of a game, so a policy can learn to think less and spend every guess. These three terms reward winning fast, give partial credit from the best guess whether or not the game finished, and penalize invalid moves. TRL logs each one separately, so `rewards/reward_win/mean` tracks the win rate.

In [ ]:
def reward_win(environments, **kwargs) -> list[float]:
    return [1.0 + 0.5 * (1 - env.num_guesses / 6) if env.won else 0.0 for env in environments]


def reward_partial(environments, **kwargs) -> list[float]:
    return [0.0 if env.won else 0.1 * env.best_greens for env in environments]


def reward_invalid(environments, **kwargs) -> list[float]:
    return [-0.1 * env.invalid_moves for env in environments]

## Create the dataset

We create a dataset with repeated prompts to control the number of training episodes.
Each entry triggers one rollout episode during training. The prompt is formatted as a chat message.

In [ ]:
from datasets import Dataset

# 150 steps x 4 prompts
dataset = Dataset.from_dict({"prompt": [[{"role": "user", "content": prompt}]] * 600})

## Set GRPO Config

Next, we define the **GRPOConfig**, which controls all key training parameters.
Each step plays 4 prompts x 16 games. Thinking stays on (`enable_thinking=True`): the SFT model writes a short reasoning before each guess, about 500 tokens per game, so `max_completion_length=4096` leaves room for 6 turns. `vllm_importance_sampling_mode="token_truncate"` clips per-token ratios instead of masking whole multi-turn sequences.

In [ ]:
from trl import GRPOConfig

model_name = "openenv/Qwen3-1.7B-Wordle-SFT"
output_dir = "Qwen3-1.7B-Wordle-SFT-GRPO"

grpo_config = GRPOConfig(
    # Training schedule / optimization
    max_steps=150,
    learning_rate=1e-6,
    lr_scheduler_type="constant",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=8,
    bf16=True,

    # GRPO configuration
    num_generations=16,
    max_completion_length=4096,
    mask_truncated_completions=True,
    max_tool_calling_iterations=6,
    vllm_importance_sampling_mode="token_truncate",
    log_completions=True,
    num_completions_to_print=1,
    chat_template_kwargs={"enable_thinking": True},

    # vLLM configuration
    use_vllm=True,
    vllm_mode="colocate",
    vllm_gpu_memory_utilization=0.35,

    # Logging / reporting
    output_dir=output_dir,
    report_to="trackio",
    logging_steps=1,
    save_steps=25,
    save_total_limit=1,

    # Memory optimization
    gradient_checkpointing=True,

    # Hub integration
    push_to_hub=True,
)

## Create the `GRPOTrainer` and start training

Now we initialize the `GRPOTrainer` with `environment_factory=WordleEnv`.

This tells the trainer to automatically handle the entire interaction loop:
- It creates a `WordleEnv` instance for each episode.
- It generates model completions, parses tool calls (like `guess`), and steps through the environment.
- It collects rewards and manages the `tool_mask` (which tokens are model-generated vs environment-generated) automatically.

No need to write a custom `rollout_func` or manage tokenization manually.

In [ ]:
from trl import GRPOTrainer

trainer = GRPOTrainer(
    model=model_name,
    reward_funcs=[reward_win, reward_partial, reward_invalid],
    train_dataset=dataset,
    args=grpo_config,
    environment_factory=WordleEnv,
)

Show memory stats before training

In [ ]:
import torch

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

And train!

In [ ]:
trainer_stats = trainer.train()

Show memory stats after training

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_training = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
training_memory_percentage = round(used_memory_for_training / max_memory * 100, 3)

print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_training} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {training_memory_percentage} %.")

## Save and push to Hub

In [ ]:
trainer.save_model(output_dir)
trainer.push_to_hub()

## Load the fine-tuned model and run inference

Now let's test our fine-tuned model by loading it and playing a game of Wordle.
We use the same `WordleEnv` class to interact with the environment, and generate model responses with standard Transformers inference.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "openenv/Qwen3-1.7B-Wordle-SFT-GRPO"  # Replace with your HF username or organization

fine_tuned_model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
import json
import re

from transformers.utils import get_json_schema

TOOL_CALL = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)
tools = [get_json_schema(WordleEnv.guess)]


def play_wordle(model, tokenizer):
    env = WordleEnv()
    # Same as training: the first observation is appended to the prompt.
    messages = [{"role": "user", "content": prompt + env.reset()}]

    while not env.done:
        prompt_text = tokenizer.apply_chat_template(
            messages, tools=tools, add_generation_prompt=True, tokenize=False, enable_thinking=True
        )
        model_inputs = tokenizer([prompt_text], return_tensors="pt").to(model.device)
        generated_ids = model.generate(**model_inputs, max_new_tokens=512, do_sample=False)
        reply = tokenizer.decode(generated_ids[0][len(model_inputs.input_ids[0]):], skip_special_tokens=True)
        call = TOOL_CALL.search(reply)
        if call is None:
            print(reply)
            break
        word = json.loads(call.group(1))["arguments"]["guess"]
        feedback = env.guess(word)
        print(f"{word}: {feedback.strip().splitlines()[-1]}")
        messages += [
            {"role": "assistant", "content": reply},
            {"role": "tool", "name": "guess", "content": feedback},
        ]

    print("Won!" if env.won else "Lost.")

Let's play the game!

In [ ]:
play_wordle(fine_tuned_model, tokenizer)